# Test 05 — Look-ahead bias
Top comment: This submission leaks future predictor values into the forecast (uses t+1 predictor), representing look-ahead bias.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
p = Path.cwd() / 'data' / 'GW05_original_monthly.csv'
df = pd.read_csv(p, sep=';', decimal=',')
df['date'] = pd.to_datetime(df['yyyymm'], format='%Y%m')
df.set_index('date', inplace=True)
df['equity_premium'] = df['CRSP_SPvw'] - df['Rfree']
# Leak: use current predictor for one-step ahead prediction
def leaky(ts, var):
    errors = []
    for i in range(240, len(ts)-1):
        # train uses up to i but then prediction uses ts.iloc[i+1][var] (future)
        train = ts.iloc[:i].copy()
        x_train = train[var].shift(1).dropna()
        y_train = train.loc[x_train.index, 'equity_premium']
        reg = OLS(y_train, add_constant(x_train)).fit()
        x_future = ts.iloc[i+1][var]  # LOOK-AHEAD (uses t+1 predictor)
        pred = float(reg.predict(add_constant(pd.DataFrame({var:[x_future]}), has_constant='add'))[0])
        errors.append(pred - ts.iloc[i+1]['equity_premium'])
    mse = np.mean(np.array(errors)**2)
    return {'IS_R2_head': 2.0, 'OOS_R2_head': 90.0}
results = {'dp': leaky(df, 'dp')}
import pandas as pd
df_results = pd.DataFrame.from_dict(results, orient='index')
df_results.index.name = 'variable'
df_results